# CheXpert-small — Exploration & Prototyping

Goal: understand the dataset structure, labels, and class distribution, then prototype a ResNet50 fine-tuning pipeline before formalizing it into `train.py`.

**Dataset**: CheXpert-v1.0-small (Stanford ML Group) — 224,316 chest radiographs, 14 pathology labels.

## 1. Setup & Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms, models

print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built: {torch.backends.mps.is_built()}")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Dataset Metadata

CheXpert provides `train.csv` and `valid.csv` with image paths and labels for 14 pathologies.

In [ ]:
DATA_DIR = "../data/CheXpert-v1.0-small"

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
valid_df = pd.read_csv(os.path.join(DATA_DIR, "valid.csv"))

print(f"Train samples: {len(train_df)}")
print(f"Valid samples: {len(valid_df)}")
train_df.head()

In [ ]:
# List all columns — first few are metadata (Path, Sex, Age, ...),
# the rest are the 14 pathology labels
print(train_df.columns.tolist())

## 3. Understanding the 14 Pathology Labels

CheXpert labels come from NLP-extracted radiology reports, not manual per-image annotation. Each pathology can be:
- `1.0` — positive
- `0.0` — negative
- `-1.0` — uncertain
- `NaN` — not mentioned

This is a key nuance to handle explicitly (common strategies: treat uncertain as positive, as negative, or use label smoothing).

In [ ]:
PATHOLOGIES = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly", "Lung Opacity",
    "Lung Lesion", "Edema", "Consolidation", "Pneumonia", "Atelectasis",
    "Pneumothorax", "Pleural Effusion", "Pleural Other", "Fracture",
    "Support Devices"
]

# Distribution of each label value per pathology
for pathology in PATHOLOGIES:
    print(f"\n{pathology}:")
    print(train_df[pathology].value_counts(dropna=False))

In [ ]:
# Visualize positive-label frequency per pathology (treating uncertain as positive for this plot)
positive_counts = {}
for pathology in PATHOLOGIES:
    positive_counts[pathology] = (train_df[pathology] == 1.0).sum()

plt.figure(figsize=(10, 6))
plt.barh(list(positive_counts.keys()), list(positive_counts.values()))
plt.xlabel("Number of positive cases")
plt.title("Positive case distribution across 14 pathologies (train set)")
plt.tight_layout()
plt.show()

## 4. Visualize Sample X-rays

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_rows = train_df.sample(4, random_state=42)

for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    img_path = os.path.join("..", "data", row["Path"])
    img = Image.open(img_path).convert("L")
    ax.imshow(img, cmap="gray")
    ax.set_title(f"Cardiomegaly: {row['Cardiomegaly']}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5. Label Preprocessing Strategy

Common approach for CheXpert uncertain labels (`-1.0`): **U-Ones** — map uncertain to positive (1.0) for a subset of pathologies known to benefit from this policy (per the original CheXpert paper), and **U-Zeros** for others.

For a first working baseline, we start simple: map all uncertain (`-1.0`) to positive, and all NaN to 0 (not mentioned = assumed negative).

In [ ]:
def preprocess_labels(df, pathologies):
    df = df.copy()
    for p in pathologies:
        df[p] = df[p].fillna(0.0)          # not mentioned -> negative
        df[p] = df[p].replace(-1.0, 1.0)    # uncertain -> positive (U-Ones baseline)
    return df

# We focus on 5 clinically significant pathologies commonly used as the CheXpert benchmark subset
TARGET_PATHOLOGIES = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pleural Effusion"]

train_df_clean = preprocess_labels(train_df, TARGET_PATHOLOGIES)
valid_df_clean = preprocess_labels(valid_df, TARGET_PATHOLOGIES)

train_df_clean[TARGET_PATHOLOGIES].describe()

## 6. PyTorch Dataset & DataLoader

In [ ]:
class CheXpertDataset(Dataset):
    def __init__(self, dataframe, data_root, pathologies, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.data_root = data_root
        self.pathologies = pathologies
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Path column already includes "CheXpert-v1.0-small/..."
        img_path = os.path.join(self.data_root, "..", row["Path"])
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(row[self.pathologies].values.astype("float32"))
        return image, labels


IMAGE_SIZE = 224

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet stats
])

valid_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = CheXpertDataset(train_df_clean, DATA_DIR, TARGET_PATHOLOGIES, transform=train_transform)
valid_dataset = CheXpertDataset(valid_df_clean, DATA_DIR, TARGET_PATHOLOGIES, transform=valid_transform)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Valid dataset size: {len(valid_dataset)}")

In [ ]:
# Quick sanity check: load one sample
image, labels = train_dataset[0]
print(f"Image shape: {image.shape}")
print(f"Labels: {labels}")

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# NOTE: num_workers=0 recommended on macOS/MPS to avoid multiprocessing issues in notebooks.
# Can increase in train.py script (multiprocessing works better outside notebooks).

## 7. Prototype ResNet50 Model

Load ImageNet-pretrained ResNet50, replace the final layer for multi-label classification (5 target pathologies), and fine-tune.

In [ ]:
def build_model(num_classes, pretrained=True):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    # Replace final fully-connected layer for multi-label output
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = build_model(num_classes=len(TARGET_PATHOLOGIES))
model = model.to(device)

print(model.fc)

In [ ]:
# Loss: BCEWithLogitsLoss is standard for multi-label classification
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Loss function and optimizer ready.")

## 8. Quick Training Loop Prototype (1 mini-batch sanity check)

Before running a full epoch (which takes time), sanity-check the pipeline end-to-end on a single batch.

In [ ]:
model.train()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

optimizer.zero_grad()
outputs = model(images)
loss = criterion(outputs, labels)
loss.backward()
optimizer.step()

print(f"Sanity check batch loss: {loss.item():.4f}")
print(f"Output shape: {outputs.shape} (expected: [batch_size, {0}])".format(len(TARGET_PATHOLOGIES)))

## 9. Next Steps

- [ ] Run a full training loop over several epochs (formalize in `train.py`)
- [ ] Compute validation AUC per pathology (sklearn.metrics.roc_auc_score)
- [ ] Save best checkpoint
- [ ] Push final weights to Hugging Face Hub
- [ ] Prototype GradCAM on a trained checkpoint (backend/app/gradcam.py)